In [1]:
import base64
import io
from datetime import datetime, timedelta
from pathlib import Path

import branca.colormap as bcm
import ee
import folium
import geemap
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import rioxarray
from PIL import Image

ModuleNotFoundError: No module named 'osgeo'

In [4]:
# Download Sufosat data
# !wget https://zenodo.org/records/15004634/files/forest-clearcuts_mainland-france_sufosat_dates_v3.tif?download=1 -O ../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3.tif
# !wget https://zenodo.org/records/15004634/files/forest-clearcuts_mainland-france_sufosat_prob_v3.tif?download=1 -O ../data/sufosat/forest-clearcuts_mainland-france_sufosat_prob_v3.tif

In [3]:
# cog_translate(
#     "../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3.tif",
#     "../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3_cog.tif",
#     cog_profiles.get("deflate")
# )

Reading input: ../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3.tif

Adding overviews...
Updating dataset tags...
Writing output to: ../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3_cog.tif


In [2]:
da = rioxarray.open_rasterio(
    "../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3_cog.tif",
    chunks={"x": 512, "y": 512},
)
print(da)
print(da.rio.crs)
print(da.rio.bounds())
print(da.attrs)

<xarray.DataArray (band: 1, y: 119159, x: 126813)> Size: 30GB
dask.array<open_rasterio-085a29630a72120494b503535ed82ec6<this-array>, shape=(1, 119159, 126813), dtype=int16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 1MB 5.164e+04 5.165e+04 ... 1.32e+06 1.32e+06
  * y            (y) float64 953kB 7.151e+06 7.151e+06 ... 5.959e+06 5.959e+06
    spatial_ref  int64 8B 0
Attributes:
    OVERVIEW_RESAMPLING:  NEAREST
    AREA_OR_POINT:        Area
    _FillValue:           0
    scale_factor:         1.0
    add_offset:           0.0
EPSG:2154
(51639.68854192982, 5959196.697153928, 1319769.6885419283, 7150786.697153928)
{'OVERVIEW_RESAMPLING': 'NEAREST', 'AREA_OR_POINT': 'Area', '_FillValue': np.int16(0), 'scale_factor': 1.0, 'add_offset': 0.0}


In [3]:
ee.Authenticate()

True


Successfully saved authorization token.


In [4]:
ee.Initialize()

Création d'une ee.Geometry() contenant la région Aquitaine (1 seul polygone)


In [5]:
ROI_FILE = Path().resolve().parent / "data" / "geometries" / "aquitaine.geojson"
roi = geemap.geojson_to_ee(str(ROI_FILE))

In [6]:
RADD_EUROPE_RESOURCE = "projects/wurnrt-raddeurope/assets/01_NRT/00_Operational/V1_IC"
radd_europe_ic = ee.ImageCollection(RADD_EUROPE_RESOURCE)
latest_radd_alert = ee.Image(
    radd_europe_ic.filterMetadata("layer", "contains", "alert")
    .sort("system:time_end", False)
    .first()
)

In [7]:
def timestamp_ms_to_yydoy(ts_ms):
    dt = datetime.utcfromtimestamp(ts_ms / 1000)
    doy = dt.timetuple().tm_yday
    return (dt.year - 2000) * 1000 + doy


def yydoy_to_datestr(yydoy):
    year = 2000 + yydoy // 1000
    doy = yydoy % 1000
    return (datetime(year, 1, 1) + timedelta(days=doy - 1)).strftime("%Y-%m-%d")


# Common range covering both datasets
date_min = 18001  # SUFOSAT starts 2018
date_max = 26097  # RADD ends ~April 2026

In [8]:
Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")
Map.add_basemap("Stadia.AlidadeSmoothDark")
Map.centerObject(roi)


alert_mask = latest_radd_alert.select("Alert").gte(2)
masked = latest_radd_alert.updateMask(alert_mask)

Map.addLayer(
    masked.select("Alert"),
    {"min": 2, "max": 3, "palette": ["cyan", "orange"]},
    "RADD Alerts confidence",
)

# Date layer
Map.addLayer(
    latest_radd_alert.select("Date"),
    {"min": date_min, "max": date_max, "palette": ["#ffffcc", "#800026"]},
    "RADD date",
)


# SUFOSAT local TIFF
Map.add_raster(
    "../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3_cog.tif",
    vmin=date_min,
    vmax=date_max,
    colormap="ylorrd",
    layer_name="SUFOSAT Date",
)


roi_fc = ee.FeatureCollection(roi)
empty_img = ee.Image().byte()
roi_contour = empty_img.paint(**{"featureCollection": roi_fc, "color": 1, "width": 3})
Map.addLayer(roi_contour, {"palette": "blue"}, "Aquitaine")


# RADD Alert confidence legend (discrete)
Map.add_legend(
    title="Alert Confidence",
    legend_dict={
        "Unconfirmed (low)": "#00FFFF",  # cyan hex
        "Confirmed (high)": "#FFA500",  # orange hex
    },
)

# Date colorbar with actual date labels
Map.add_colorbar(
    vis_params={"min": date_min, "max": date_max, "palette": ["#ffffcc", "#800026"]},
    label=f"Alert Date ({yydoy_to_datestr(date_min)} → {yydoy_to_datestr(date_max)})",
    orientation="horizontal",
)

Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [2]:
deps = gpd.read_file("../data/geometries/all_departements.geojson")
gironde = deps[deps["code"] == "33"]

ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /home/florent/miniconda3/lib/gdalplugins/.././libcurl.so.4)
ERROR 1: /lib/x86_64-linux-gnu/libssl.so.3: version `OPENSSL_3.2.0' not found (required by /ho

In [3]:
# gironde_wgs84 = gironde.to_crs('EPSG:4326')
# gironde_ee = geemap.geopandas_to_ee(gironde_wgs84)

# radd_date = latest_radd_alert.select('Date').updateMask(
#     latest_radd_alert.select('Alert').gte(2)
# )

# task = ee.batch.Export.image.toDrive(
#     image=radd_date,
#     description='radd_gironde',
#     folder='GEE_exports',
#     scale=10,
#     region=gironde_ee.geometry(),
#     crs='EPSG:2154',
#     fileFormat='GeoTIFF',
#     maxPixels=1e10
# )
# task.start()

# import time

# while True:
#     status = task.status()
#     print(status['state'], status.get('progress', ''))
#     if status['state'] in ('COMPLETED', 'FAILED', 'CANCELLED'):
#         break
#     time.sleep(30)

In [4]:
def timestamp_ms_to_yydoy(ts_ms):
    dt = datetime.utcfromtimestamp(ts_ms / 1000)
    doy = dt.timetuple().tm_yday
    return (dt.year - 2000) * 1000 + doy


def yydoy_to_datestr(yydoy):
    year = 2000 + yydoy // 1000
    doy = yydoy % 1000
    return (datetime(year, 1, 1) + timedelta(days=doy - 1)).strftime("%Y-%m-%d")


# Common range covering both datasets
date_min = 18001  # SUFOSAT starts 2018
date_max = 26097  # RADD ends ~April 2026


def raster_to_base64_png(da, vmin, vmax, colormap_name):
    da_wgs84 = da.rio.reproject("EPSG:4326")
    data = da_wgs84.squeeze().values.astype(float)
    nodata = da_wgs84.rio.nodata
    if nodata is not None:
        data[data == nodata] = np.nan
    data[(data < vmin) | (data > vmax)] = np.nan  # mask out-of-range as transparent

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    rgba = cm.get_cmap(colormap_name)(norm(data))
    rgba[np.isnan(data)] = [0, 0, 0, 0]

    img = Image.fromarray((rgba * 255).astype(np.uint8))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


sufosat = rioxarray.open_rasterio(
    "../data/sufosat/forest-clearcuts_mainland-france_sufosat_dates_v3_cog.tif",
    chunks={"x": 512, "y": 512},
)
radd = rioxarray.open_rasterio(
    "../data/radd_europe/radd_gironde.tif", chunks={"x": 512, "y": 512}
)


# Clip SUFOSAT (EPSG:2154) — reproject geometry to match raster CRS
gironde_2154 = gironde.to_crs("EPSG:2154")

centroid = gironde_2154.geometry.centroid.iloc[0]
cx, cy = centroid.x, centroid.y
half = 50_000

bbox = dict(minx=cx - half, maxx=cx + half, miny=cy - half, maxy=cy + half)

sufosat_box = sufosat.rio.clip_box(**bbox).compute()
radd_box = radd.rio.clip_box(**bbox).compute()

sufosat_png = raster_to_base64_png(sufosat_box, date_min, date_max, "YlOrRd")
radd_png = raster_to_base64_png(radd_box, date_min, date_max, "YlOrRd")


sufosat_wgs84 = sufosat_box.rio.reproject("EPSG:4326")
l, b, r, t = sufosat_wgs84.rio.bounds()
bounds = [[b, l], [t, r]]


left, bottom, right, top = gironde_2154.total_bounds
sufosat_gironde = sufosat.rio.clip_box(
    minx=left, miny=bottom, maxx=right, maxy=top
).rio.clip(  # Runs a first cheap pre-clip so that the final clip is done on a much smaller chunk of the raster
    gironde_2154.geometry
)


centroid_wgs84 = gironde.to_crs("EPSG:4326").geometry.centroid.iloc[0]

m = folium.Map(location=[centroid_wgs84.y, centroid_wgs84.x], zoom_start=9)

# folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="ESRI Imagery",
).add_to(m)

folium.raster_layers.ImageOverlay(
    image=f"data:image/png;base64,{sufosat_png}",
    bounds=bounds,
    opacity=0.8,
    name="SUFOSAT Date",
).add_to(m)

folium.raster_layers.ImageOverlay(
    image=f"data:image/png;base64,{radd_png}",
    bounds=bounds,
    opacity=0.8,
    name="RADD Date",
).add_to(m)

folium.GeoJson(
    gironde.to_crs("EPSG:4326"),
    name="Gironde boundary",
    style_function=lambda _: {"fillOpacity": 0, "color": "black", "weight": 2},
).add_to(m)


colormap = bcm.LinearColormap(
    colors=["#ffffcc", "#fd8d3c", "#800026"],
    vmin=date_min,
    vmax=date_max,
    caption=f"Clear-cut date ({yydoy_to_datestr(date_min)} → {yydoy_to_datestr(date_max)})",
)
colormap.add_to(m)
m.get_root().html.add_child(
    folium.Element("""                                                                                                                                        
<style>                                                                                                                                                                               
  .colormap { background: rgba(255,255,255,0.85); padding: 8px; border-radius: 4px; }                                                                                                 
  .colormap .caption { color: #222 !important; font-size: 13px !important; font-weight: bold !important; }                                                                            
</style>                                                                                                                                                                              
""")
)


folium.LayerControl().add_to(m)
m.save("gironde_clearcuts.html")
print("Saved.")

/tmp/ipykernel_197337/406337793.py:25: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = cm.get_cmap(colormap_name)(norm(data))
/tmp/ipykernel_197337/406337793.py:25: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgba = cm.get_cmap(colormap_name)(norm(data))
/tmp/ipykernel_197337/406337793.py:74: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid_wgs84 = gironde.to_crs('EPSG:4326').geometry.centroid.iloc[0]


Saved.
